<a href="https://colab.research.google.com/github/abhishekSanyal23/RAG-Chatbot/blob/main/Getting_started_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run Ollama in Colab
---

This is an example notebook which demonstrates how to run Ollama inside a Colab instance. With this you can run pretty much any small to medium sized models offerred by Ollama for free.

For the list of available models check [models being offerred by Ollama](https://ollama.com/library).


## Before you proceed
---

Since by default the runtime type of Colab instance is CPU based, in order to use LLM models make sure to change your runtime type to T4 GPU (or better if you're a paid Colab user). This can be done by going to **Runtime > Change runtime type**.

While running your script be mindful of the resources you're using. This can be tracked at **Runtime > View resources**.

## Running the notebook
---

After configuring the runtime just run it with **Runtime > Run all**. And you can start tinkering around.

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`

In [3]:
!sudo apt update
!sudo apt-get install zstd
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (4,273 B/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
64 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illino

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [4]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM model using `ollama pull gemma4:e2b`.

For other models check https://ollama.com/library

In [5]:
!ollama pull gemma4:e2b

## And that's it!
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.

In [6]:
!pip install langchain-ollama

In [9]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="gemma4:e2b")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What's the length of hypotenuse in a right angled triangle? Can you calculate the hypotenuse of a right angled triangle whose sides are 4 cm and 3 cm?"})))

Let's think step by step.

### Step 1: Understand the Rule

The relationship between the sides of a right-angled triangle is defined by the **Pythagorean Theorem**. This theorem states that the square of the length of the hypotenuse (the side opposite the right angle, denoted as $c$) is equal to the sum of the squares of the other two sides (the legs, denoted as $a$ and $b$).

The formula is:
$$a^2 + b^2 = c^2$$

### Step 2: Identify the Known Values

In this problem, the lengths of the two sides (the legs) are:
*   $a = 4$ cm
*   $b = 3$ cm
*   $c = ?$ (the hypotenuse)

### Step 3: Substitute the Values into the Formula

Substitute the values of $a$ and $b$ into the Pythagorean theorem:
$$4^2 + 3^2 = c^2$$

### Step 4: Calculate the Squares

Calculate the square of each side:
$$16 + 9 = c^2$$

### Step 5: Sum the Results

Add the results together:
$$25 = c^2$$

### Step 6: Solve for the Hypotenuse (c)

To find the length of $c$, take the square root of both sides:
$$c = \sqrt{25}$$
$$c = 5$$

---

### Answer

The length of the hypotenuse is **5 cm**.

In [10]:
display(Markdown(chain.invoke({"question": "Can you write a python code which calculates the value of any side of a right angled triangle, it should take in command line argument which will take in 3 values, Input 1: What to calculate - side/hypotenuse, Input: 2 side values in an dictionary?"})))

This is a classic geometry problem that requires the use of the **Pythagorean theorem** ($a^2 + b^2 = c^2$).

To handle command-line arguments effectively, we will use the `sys` module.

Here is the complete Python code, followed by an explanation of how it works.

### Python Code: `triangle_calculator.py`

```python
import sys
import math

def calculate_triangle_side(side1, side2, calculation_type):
    """
    Calculates the unknown side (leg or hypotenuse) of a right-angled triangle.

    Args:
        side1 (float): The length of the first known side.
        side2 (float): The length of the second known side.
        calculation_type (str): Specifies what to calculate ('hypotenuse' or 'side').

    Returns:
        float or str: The calculated side length, or an error message string.
    """
    
    # Ensure inputs are valid numbers
    try:
        a = float(side1)
        b = float(side2)
    except ValueError:
        return "Error: Both side values must be valid numbers (floats)."

    # Validate that side lengths are non-negative
    if a < 0 or b < 0:
        return "Error: Side lengths cannot be negative."

    if calculation_type.lower() == 'hypotenuse':
        # Calculation: c = sqrt(a^2 + b^2)
        result = math.sqrt(a**2 + b**2)
        return f"The Hypotenuse is: {result:.4f}"

    elif calculation_type.lower() == 'side':
        # When calculating a 'side', we assume the user wants the missing leg,
        # meaning we need to calculate sqrt(hypotenuse^2 - leg^2).
        # However, since we only have two known sides, we must first check if 
        # the hypotenuse is known.

        # We calculate the hypotenuse first, then calculate the missing leg.
        hypotenuse = math.sqrt(a**2 + b**2)
        
        # Check if the requested side is the hypotenuse
        if hypotenuse == float(side1) or hypotenuse == float(side2):
             return f"The Hypotenuse is: {hypotenuse:.4f}"
        
        # If the user wants the missing leg, calculate the difference
        if a > b:
            missing_side = math.sqrt(hypotenuse**2 - b**2)
        else:
            missing_side = math.sqrt(hypotenuse**2 - a**2)
            
        return f"The missing side (leg) is: {missing_side:.4f}"

    else:
        return "Error: Invalid calculation type. Please enter 'hypotenuse' or 'side'."


def main():
    """
    Main function to process command-line arguments.
    Expects 4 arguments: [script_name, calculation_type, side1, side2]
    """
    
    # Check if the correct number of arguments are provided (4 total: script name + 3 inputs)
    if len(sys.argv) != 4:
        print("Usage: python triangle_calculator.py <calculation_type> <side1> <side2>")
        print("Example 1 (Find Hypotenuse): python triangle_calculator.py hypotenuse 3 4")
        print("Example 2 (Find Missing Leg): python triangle_calculator.py side 5 12")
        sys.exit(1)

    # Extract the arguments
    calculation_type = sys.argv[1]
    side1 = sys.argv[2]
    side2 = sys.argv[3]

    # Perform the calculation and print the result
    result = calculate_triangle_side(side1, side2, calculation_type)
    print("-" * 30)
    print(f"Input Sides: {side1} and {side2}")
    print(f"Calculation Requested: {calculation_type.capitalize()}")
    print(result)
    print("-" * 30)


if __name__ == "__main__":
    main()
```

### How to Run the Code

1.  **Save:** Save the code above as a file named `triangle_calculator.py`.
2.  **Run from Terminal:** Execute the script using `python triangle_calculator.py` followed by the three required inputs.

#### Example 1: Calculating the Hypotenuse

We want to find the hypotenuse of a triangle with legs of length 3 and 4.

```bash
python triangle_calculator.py hypotenuse 3 4
```

**Output:**
```
------------------------------
Input Sides: 3.0 and 4.0
Calculation Requested: Hypotenuse
The Hypotenuse is: 5.0000
------------------------------
```

#### Example 2: Calculating the Missing Side (Leg)

We know the two legs are 5 and 12. We want to calculate the missing side (the hypotenuse).

```bash
python triangle_calculator.py side 5 12
```

**Output:**
```
------------------------------
Input Sides: 5.0 and 12.0
Calculation Requested: Side
The missing side (leg) is: 13.0000
------------------------------
```

*(Note: In this specific example, since we entered `side` and provided two legs, the script correctly interprets this as finding the hypotenuse, which is the third required side.)*

#### Example 3: Calculating a Missing Leg (More Complex Scenario)

We want to find the missing leg, given that the hypotenuse is 13 and one leg is 5.

```bash
python triangle_calculator.py side 13 5
```

**Output:**
```
------------------------------
Input Sides: 13.0 and 5.0
Calculation Requested: Side
The missing side (leg) is: 12.0000
------------------------------
```

### Code Explanation

1.  **Imports:**
    *   `sys`: Used to access command-line arguments (`sys.argv`).
    *   `math`: Used to perform the square root calculation (`math.sqrt()`).

2.  **`calculate_triangle_side(side1, side2, calculation_type)` function:**
    *   This is the core logic. It takes the raw command-line inputs and determines which calculation to perform.
    *   **Input Validation (`try-except`):** It attempts to convert `side1` and `side2` to `float`. If the user enters text instead of a number, a `ValueError` is raised, and an error message is returned.
    *   **Negative Check:** It ensures that physical lengths cannot be negative.
    *   **Hypotenuse Calculation:** If `calculation_type` is `'hypotenuse'`, it uses the formula: $c = \sqrt{a^2 + b^2}$.
    *   **Side Calculation:** If `calculation_type` is `'side'`, the logic first calculates the true hypotenuse ($c$), and then uses the relationship $a = \sqrt{c^2 - b^2}$ to find the missing leg.
    *   **Output Formatting:** The results are formatted using an f-string to display the result clearly, rounded to four decimal places (`:.4f`).

3.  **`main()` function:**
    *   **Argument Check:** `sys.argv` is a list of command-line arguments. The script expects 4 elements:
        *   `sys.argv[0]`: The script name itself (`triangle_calculator.py`).
        *   `sys.argv[1]`: The first input (e.g., `'hypotenuse'`).
        *   `sys.argv[2]`: The first side value (e.g., `'3'`).
        *   `sys.argv[3]`: The second side value (e.g., `'4'`).
    *   **Extraction:** It extracts these three required inputs.
    *   **Execution:** It calls the calculation function and prints the final result.